<a href="https://colab.research.google.com/github/Afique1/Business-Review-by-user/blob/Afique's-work/2.3_Indigenous_Education_Report_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Strategy:
1. get the relevent libs, and the PDF
2. check for all the different elements in the pdf
3. the goal is to generate insights from the pdf, by extracting figures, tables.
    - read tables through Tabula
    - figures (as images) with pillow (PIL)

Getting the libraries

In [ ]:
! pip install pdfminer.six # extract and process text from a pdf file
! pip install wget # for downloading online files
! pip install PyMuPDF # processing PDF
! pip install tabula-py # for extracting tables from pdf

In [2]:
import re
import pandas as pd

from pdfminer.high_level import extract_text, extract_pages
import tabula
import io # for i/o of images
import PIL.Image # Python Imaging Library (PIL); pillow
import fitz # PyMuPDF
# import pytesseract # OCR


getting the pdf

In [3]:
import wget

link = "https://github.com/Afique1/Business-Review-by-user/raw/main/UA_Indigenous_Strategy_Annual_Report_May-2022.pdf"
pdf = wget.download(link)

explore the texts and pages

In [4]:
from pdfminer.high_level import extract_pages

layout_types = set()

for page_layout in extract_pages(pdf):
    for element in page_layout:
        layout_types.add(type(element).__name__)

layout_types_list = list(layout_types)
print("Layout types found in the PDF:")
print(layout_types_list)

print("\nTotal pages in the PDF:", len(pdf))

Layout types found in the PDF:
['LTTextBoxHorizontal', 'LTLine', 'LTTextLineHorizontal', 'LTRect', 'LTCurve', 'LTFigure']

Total pages in the PDF: 49


Select the drawings (charts, bars)

In [5]:
import fitz # Import PyMuPDF

pdf_document = fitz.open(pdf)
drawing_count = 0

print("Information about the first 5 drawings found in the PDF:")
for page_num in range(len(pdf_document)):
    page = pdf_document.load_page(page_num)
    drawings = page.get_drawings()

    for drawing in drawings:
        print(drawing)
        drawing_count += 1
        if drawing_count >= 5:
            break
    if drawing_count >= 5:
        break

pdf_document.close()

Information about the first 5 drawings found in the PDF:
{'items': [('re', Rect(-8.503999710083008, -8.50396728515625, 603.7789916992188, 850.39404296875), 1)], 'type': 'f', 'even_odd': False, 'fill_opacity': 1.0, 'fill': (0.08198672533035278, 0.1449912190437317, 0.3019913136959076), 'rect': Rect(-8.503999710083008, -8.50396728515625, 603.7789916992188, 850.39404296875), 'seqno': 0, 'layer': '', 'closePath': None, 'color': None, 'width': None, 'lineCap': None, 'lineJoin': None, 'dashes': None, 'stroke_opacity': None}
{'items': [('c', Point(70.6488037109375, 742.689697265625), Point(70.59180450439453, 742.6817016601562), Point(70.53580474853516, 742.7166748046875), Point(70.5038070678711, 742.7686767578125)), ('c', Point(70.5038070678711, 742.7686767578125), Point(69.43780517578125, 744.522705078125), Point(68.42980194091797, 746.3427124023438), Point(67.50780487060547, 748.1787109375)), ('c', Point(67.50780487060547, 748.1787109375), Point(67.48180389404297, 748.231689453125), Point(67

In [6]:
doc = fitz.open(pdf)
page = doc[14] # Get the first page

drawings = page.get_drawings()

# Iterate through the extracted drawing objects
for drawing in drawings:
    print(f"Drawing Type: {drawing.get('type')}")
    print(f"Bounding Box: {drawing.get('rect')}")
    print(f"Line Width: {drawing.get('width')}")
    # The 'items' list contains the actual drawing instructions (vectors)
    # print(drawing.get('items'))

[('c', Point(537.9536743164062, 100.31890869140625), Point(537.0926513671875, 93.28290557861328), Point(535.3136596679688, 86.4059066772461), Point(532.5156860351562, 79.29791259765625)), ('c', Point(532.5156860351562, 79.29791259765625), Point(532.4986572265625, 79.25590515136719), Point(532.4646606445312, 79.22691345214844), Point(532.4226684570312, 79.21791076660156)), ('c', Point(532.4226684570312, 79.21791076660156), Point(532.3816528320312, 79.20890808105469), Point(532.3396606445312, 79.22090911865234), Point(532.3096923828125, 79.25190734863281)), ('c', Point(532.3096923828125, 79.25190734863281), Point(531.169677734375, 80.41690826416016), Point(529.7467041015625, 81.91490936279297), Point(528.8496704101562, 83.17291259765625)), ('c', Point(528.8496704101562, 83.17291259765625), Point(528.8197021484375, 83.21490478515625), Point(528.815673828125, 83.27090454101562), Point(528.8396606445312, 83.31690979003906)), ('c', Point(528.8396606445312, 83.31690979003906), Point(528.90667

In [33]:
import fitz
import PIL.Image
import io

def extract_and_save_figure_with_drawing(pdf_path, figure_num):
    """
    Finds a figure by its title, calculates a bounding box from the title
    to the end of the associated drawing, and saves that area as an image.

    Args:
        pdf_path (str): The path to the PDF file.
        figure_num (str): The title of the figure to find (e.g., "Figure 1").

    Returns:
        tuple: A tuple containing the found figure title (or None if not found)
               and a boolean indicating if an image was successfully saved. Returns
               (None, False) if the figure title or associated drawing is not found.
    """
    doc = fitz.open(pdf_path)
    figure_found = False
    image_saved = False
    found_title = None

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()

        # Search for the figure title on the page
        text_instances = page.search_for(figure_num)
        # text_instances_with_colon = figure_title+':' # this is to get only from the

        if text_instances:
            print(f"Found '{figure_num}': on page {page_num + 1}")
            figure_found = True
            found_title = figure_num

            figure_text_bbox = text_instances[0]
            print(f"Figure text bounding box: {figure_text_bbox}")

            # Attempt to find drawings on this page that could be the figure
            drawings = page.get_drawings()

            if drawings:
                print(f"Found {len(drawings)} drawing(s) on page {page_num + 1}.")

                # Find a drawing whose bounding box is below the text's bounding box
                # This heuristic might need refinement depending on the PDF layout.
                associated_drawing = None
                for drawing in drawings:
                    # Simple heuristic: check if the drawing's top is below the text's bottom
                    # and if there's horizontal overlap. This might need refinement.
                     if drawing['rect'].y0 > figure_text_bbox.y1 and \
                           max(drawing['rect'].x0, figure_text_bbox.x0) < min(drawing['rect'].x1, figure_text_bbox.x1):
                            associated_drawing = drawing
                            break

                if associated_drawing:
                    print(f"Identified a potential associated drawing with bounding box: {associated_drawing['rect']}")

                    # Calculate a new bounding box from the bottom of the text bbox
                    # to the bottom of the associated drawing bbox, spanning the width
                    # of the drawing.
                    clip_bbox = fitz.Rect(
                        min(figure_text_bbox.x0, associated_drawing['rect'].x0), # Use the leftmost extent of text or drawing
                        figure_text_bbox.y0, # Start from the top of the text bounding box
                        max(figure_text_bbox.x1, associated_drawing['rect'].x1), # Use the rightmost extent of text or drawing
                        associated_drawing['rect'].y1 # Extend to the bottom of the drawing bounding box
                    )
                    print(f"Calculated clipping bounding box: {clip_bbox}")

                    # Create a filename using the figure title, replacing spaces and colons
                    filename = f"{figure_num.replace(' ', '_').replace(':', '')}_drawing.png"


                    try:
                        # Render the calculated area as a pixmap
                        pix = page.get_pixmap(clip=clip_bbox)
                        img = PIL.Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                        img.save(filename)
                        print(f"Saved combined area for '{figure_num}': as {filename}")
                        image_saved = True
                    except Exception as e:
                        print(f"Could not render and save the combined area for '{figure_num}': {e}")

                else:
                     print(f"Could not find an associated drawing close to the text for '{figure_num}'. Cannot save the combined area.")


            else:
                print(f"No drawings found on page {page_num + 1}. Cannot save the combined area.")

            break # Stop searching after finding the figure title

    doc.close()

    if not figure_found:
        print(f"'{figure_num}': not found in the PDF text.")
        return None, False # Explicitly return tuple when figure not found

    return found_title, image_saved

# Example usage: [we will use figure_data dictionary to get the drawings]
# Call the function with the PDF path and the figure title you want to extract
# Make sure the 'pdf' variable is defined and holds the path to your PDF
# extracted_title, saved_status = extract_and_save_figure_with_drawing(pdf, "Figure 1")
# if saved_status:
#     print(f"Successfully processed combined area for: {extracted_title}")

# extracted_title, saved_status = extract_and_save_figure_with_drawing(pdf, "Figure 8")
# if saved_status:
#      print(f"Successfully processed combined area for: {extracted_title}")

Use the `extract_and_save_figure_with_drawing` function to extract the drawing as images (used later after creating `figure_data` dictionary)


---



Select the keywords for Figure

In [11]:
pdf_text = extract_text(pdf)

In [12]:
# Now that the txt's have been extracted from the pdf
# let's find the matches with figure keyword

# Initialize a dictionary to store figure data
figure_data = {}

# Find all occurrences of "Figure" followed by a number and label
# This regex captures the figure number in group 1 and the label in group 2
figure_matches = re.findall(r'Figure\s+(\d+):\s*(.*)', pdf_text)

if figure_matches:
    print("Found figures and their potential labels:")
    for fig_number, fig_label in figure_matches:
        figure_key = f"Figure {fig_number}"

        # save in the dict
        figure_data[figure_key] = {'title': fig_label.strip()}
        print(f"- {figure_key}: {figure_data[figure_key]['title']}")
else:
    print("No figures with labels found in the text.")

Found figures and their potential labels:
- Figure 1: Indigenous student enrolments, 2006 to 2020
- Figure 2: Share of Indigenous student enrolments, 2006 to 2020
- Figure 3: Annual growth in Indigenous student enrolments, 2007 to 2020
- Figure 4: Enrolments by broad disciplines, 2020
- Figure 5: Annual growth in undergraduate applications, 2013 to 2021
- Figure 6: Share of undergraduate applications, by age, 2021
- Figure 7: Share of Indigenous undergraduate applications compared to share of Indigenous
- Figure 8: Share of undergraduate applications, by gender, 2021
- Figure 9: Share of Indigenous undergraduate applications by broad disciplines, 2012, 2020 and 2021
- Figure 10: Number of award course completions by Indigenous students, by course level
- Figure 11: Nine-year completion rates of commencing Indigenous and non-Indigenous Bachelor
- Figure 12: Share of Indigenous students commencing a Bachelor degree that never return – after
- Figure 13: Retention and success rates of dom

In [34]:
# Iterate through the figure_data dictionary and extract/save each figure drawing using the updated function
for figure_num in figure_data.keys():
    extracted_title, saved_status = extract_and_save_figure_with_drawing(pdf, figure_num)
    if saved_status:
        print(f"Successfully processed combined area for: {extracted_title}")
    else:
        print(f"Could not process combined area for: {figure_num}")
    print("-" * 40)

Found 'Figure 1': on page 9
Figure text bounding box: Rect(56.692901611328125, 242.2684326171875, 97.26199340820312, 254.8414306640625)
Found 35 drawing(s) on page 9.
Identified a potential associated drawing with bounding box: Rect(56.69300079345703, 264.0989990234375, 538.5830078125, 465.6240234375)
Calculated clipping bounding box: Rect(56.692901611328125, 242.2684326171875, 538.5830078125, 465.6240234375)
Saved combined area for 'Figure 1': as Figure_1_drawing.png
Successfully processed combined area for: Figure 1
----------------------------------------
Found 'Figure 2': on page 9
Figure text bounding box: Rect(445.05218505859375, 212.71142578125, 481.74267578125, 224.84442138671875)
Found 35 drawing(s) on page 9.
Identified a potential associated drawing with bounding box: Rect(56.69300079345703, 264.0989990234375, 538.5830078125, 465.6240234375)
Calculated clipping bounding box: Rect(56.69300079345703, 212.71142578125, 538.5830078125, 465.6240234375)
Saved combined area for 'Fig

In [13]:
# # Iterate through the figure_data dictionary and extract/save each figure drawing
# for figure_title in figure_data.keys():
#     extract_and_save_figure_drawing(pdf, figure_title)
#     print(f"Successfully processed: {figure_title}")
#     print("-"*40)


Found 'Figure 1' on page 9
Found 35 drawing(s) on page 9.
Figure text bounding box: Rect(56.692901611328125, 242.2684326171875, 97.26199340820312, 254.8414306640625)
Identified a potential drawing for 'Figure 1' with bounding box: Rect(56.69300079345703, 264.0989990234375, 538.5830078125, 465.6240234375)
Saved drawing for 'Figure 1' as Figure_1.png
Successfully processed: Figure 1
----------------------------------------
Found 'Figure 2' on page 9
Found 35 drawing(s) on page 9.
Figure text bounding box: Rect(445.05218505859375, 212.71142578125, 481.74267578125, 224.84442138671875)
Identified a potential drawing for 'Figure 2' with bounding box: Rect(56.69300079345703, 264.0989990234375, 538.5830078125, 465.6240234375)
Saved drawing for 'Figure 2' as Figure_2.png
Successfully processed: Figure 2
----------------------------------------
Found 'Figure 3' on page 10
Found 14 drawing(s) on page 10.
Figure text bounding box: Rect(56.692901611328125, 256.2684326171875, 97.26199340820312, 268.

Figure data with content: (not important)

In [14]:
# Now that the txt's have been extracted from the pdf
# let's find the matches with figure keyword

# Initialize a dictionary to store figure data
figure_data = {}

# Find all occurrences of "Figure" followed by a number and label
# This regex captures the figure number in group 1 and the label in group 2
figure_matches = re.findall(r'Figure\s+(\d+):\s*(.*)', pdf_text)

if figure_matches:
    # Find all occurrences of "Figure" followed by a number and the subsequent text up to "Source"
    # This regex captures the figure number in group 1 and the text in group 2
    figure_content_matches = re.findall(r'Figure\s+\d+:\s*.*?(\n.*?)Source', pdf_text, re.DOTALL)

    for i, (fig_number, fig_label) in enumerate(figure_matches):
        figure_key = f"Figure {fig_number}"

        # save in the dict
        figure_data[figure_key] = {'title': fig_label.strip()}

        # Add the extracted content if available
        if i < len(figure_content_matches):
            figure_data[figure_key]['content'] = figure_content_matches[i].strip()

else:
    print("No figures with labels found in the text.")

In [15]:
print("\nFigure Data Dictionary:")
print(figure_data)


Figure Data Dictionary:
{'Figure 1': {'title': 'Indigenous student enrolments, 2006 to 2020', 'content': '22,897\n\n21,033\n\n19,237\n\n19,935\n\n17,800\n\n16,108\n\n15,043\n\n13,723\n\n12,595\n\n11,753\n\n8,816\n\n9,329\n\n9,490\n\n10,400\n\n11,024\n\n2006\n\n2007\n\n2008\n\n2009\n\n2010\n\n2011\n\n2012\n\n2013\n\n2014\n\n2015\n\n2016\n\n2017\n\n2018\n\n2019\n\n2020'}, 'Figure 2': {'title': 'Share of Indigenous student enrolments, 2006 to 2020', 'content': '2.04%\n\n1.95%\n\n1.86%\n\n1.80%\n\n1.69%\n\n1.22% 1.25% 1.25%\n\n1.30% 1.30% 1.34% 1.37% 1.41%\n\n1.56%\n\n1.48%\n\n2006\n\n2007\n\n2008\n\n2009\n\n2010\n\n2011\n\n2012\n\n2013\n\n2014\n\n2015\n\n2016\n\n2017\n\n2018\n\n2019\n\n2020'}, 'Figure 3': {'title': 'Annual growth in Indigenous student enrolments, 2007 to 2020', 'content': 'Total domestic enrolments\n\nIndigenous students\n\nNon-Indigenous students\n\n12.0%\n\n10.0%\n\n8.0%\n\n6.0%\n\n4.0%\n\n2.0%\n\n0.0%\n\n2007\n\n2008\n\n2009\n\n2010\n\n2011\n\n2012\n\n2013\n\n2014\n\n

get the tables

In [18]:
tables = tabula.read_pdf(pdf, pages='all')
print("Total Number of tabular formatted data: ",len(tables))


Total Number of tabular formatted data:  28


In [19]:
# for table in tables:
#     print(table)

get the images [not important, img are just img of peoples]

In [20]:
# from pdfminer.high_level import extract_pages
# from pdfminer.layout import LTFigure

# print("Extracting LTFigure elements:")
# for page_layout in extract_pages(pdf):
#     for element in page_layout:
#         if isinstance(element, LTFigure):
#             print(f"Found LTFigure on page {page_layout.pageid} at bounding box: {element.bbox}")
#             # You could add code here to save the image if needed

Extracting LTFigure elements:
Found LTFigure on page 1 at bounding box: (0.0, 841.89, 595.276, 841.89)
Found LTFigure on page 1 at bounding box: (244.5562134, -1.0311035, 596.3372779, 756.7086792)
Found LTFigure on page 2 at bounding box: (0.0, 841.89, 595.276, 841.89)
Found LTFigure on page 3 at bounding box: (0.0, 841.89, 595.276, 841.89)
Found LTFigure on page 3 at bounding box: (0.0, 841.89, 595.276, 841.89)
Found LTFigure on page 4 at bounding box: (0.0, 841.89, 595.276, 841.89)
Found LTFigure on page 4 at bounding box: (0.0, 841.89, 595.276, 841.89)
Found LTFigure on page 4 at bounding box: (387.6411438, 143.3214713, 538.8208276, 299.6642456)
Found LTFigure on page 4 at bounding box: (222.327301, 120.5855921, 373.49888350000003, 299.6238404)
Found LTFigure on page 4 at bounding box: (57.0449791, 72.478826, 208.2725967, 299.5518799)
Found LTFigure on page 6 at bounding box: (303.6948762, 304.2065009, 539.6145116, 559.2143553999999)
Found LTFigure on page 7 at bounding box: (303.69

the following code contains the images of the people/

In [21]:
# pdf_figures = fitz.open(pdf)
# counter = 1

# for i in range(len(pdf_figures)):
#     page = pdf_figures[i]
#     images = page.get_images()

#     for image in images:

#         base_image = pdf_figures.extract_image(image[0])
#         image_data = base_image["image"]
#         img = PIL.Image.open(io.BytesIO(image_data))
#         extension = base_image["ext"]
#         img.save(open(f"image{counter}.{extension}", "wb"))
#         counter += 1
#         img.show()


In [22]:
# len(pdf_figures)